# Google ad news: every step of the pipeline

One section per script, in the order they run. The `.py` scripts are the originals; this notebook is a copy made by `to_notebooks.py`. See `README.md` for what each step does and why.

| Step | Script |
| --- | --- |
| 1 | `google_ad_news.py` |
| 2 | `fetch_summaries.py` |
| 3 | `clean_news.py` |
| 4 | `fetch_leads.py` |
| 5 | `analyze_news.py` |
| 6 | `scan_topics.py` |

How to use it:
- Start Jupyter from this folder, so `data/...` points to the data of the assignment.
- Every script has a `main()`, so run a step's cells from the top of its section, not only its last cell.
- Steps that go to the web have a warning above their `main()` cell.

## Step 1: `google_ad_news.py`

Collect news headlines about Google's ad revenue trend from Google News.

Google News returns at most ~100 results per search, so the script splits the
period into small time windows and runs one search per window. The results
are merged, de-duplicated and saved as a CSV file.

### Imports

In [ ]:
import argparse
import time
import xml.etree.ElementTree as ET
from datetime import date, timedelta
from email.utils import parsedate_to_datetime
from pathlib import Path

import pandas as pd
import requests

### Constants

In [ ]:
RSS_URL = "https://news.google.com/rss/search"
# Google News drops whatever comes after roughly 32 words, including the date
# limits, so the search is split into several short queries instead of one long
# one. intitle:Google keeps only headlines that name Google.
QUERIES = [
    'intitle:Google "ad revenue" search traffic',
    'intitle:Google "search revenue" (traffic OR queries OR usage OR "market share")',
    'intitle:Google search (traffic OR queries OR usage) (ads OR advertising) revenue',
    'intitle:Google "search market share" revenue',
]
AI_QUERY = '(AI OR ChatGPT OR "AI Overviews")'
HEADERS = {"User-Agent": "Mozilla/5.0 (strategic-scanning coursework script)"}

### Functions

In [ ]:
def time_windows(years, window_days):
    """Yield (start, end) dates covering the last `years` years, oldest first."""
    today = date.today()
    start = today - timedelta(days=365 * years)
    while start < today:
        end = min(start + timedelta(days=window_days), today)
        yield start, end
        start = end

In [ ]:
def fetch_window(query, start, end):
    """Run one Google News search limited to [start, end) and return its articles."""
    params = {
        "q": f"after:{start} before:{end} {query}",
        "hl": "en-US",
        "gl": "US",
        "ceid": "US:en",
    }
    response = requests.get(RSS_URL, params=params, headers=HEADERS, timeout=30)
    response.raise_for_status()

    articles = []
    for item in ET.fromstring(response.content).iter("item"):
        source = item.findtext("source", default="")
        title = item.findtext("title", default="")
        # Google News titles look like "Headline - Source name"
        if source and title.endswith(f" - {source}"):
            title = title[: -len(source) - 3]
        published = parsedate_to_datetime(item.findtext("pubDate"))
        # Skip anything outside the window, in case Google ignored the date limits
        if not start - timedelta(days=2) <= published.date() <= end + timedelta(days=2):
            continue
        articles.append(
            {
                "published": published,
                "title": title,
                "source": source,
                "link": item.findtext("link", default=""),
                "query": query,
            }
        )
    return articles

### main(): the steps, in the order they run

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Collect news headlines about Google's ad revenue trend from Google News.")
    parser.add_argument("--ai", action="store_true", help="only keep articles that also mention AI")
    parser.add_argument("--query", help="use this single search query instead of the default ones")
    parser.add_argument("--years", type=int, default=3, help="how many years back to search (default: 3)")
    parser.add_argument("--window-days", type=int, default=30, help="size of each search window (default: 30)")
    parser.add_argument("--out", help="output CSV path (default: data/google_ad_news[_ai].csv)")
    args = parser.parse_args(shlex.split(OPTIONS))

    queries = [args.query] if args.query else QUERIES
    if args.ai:
        queries = [f"{query} {AI_QUERY}" for query in queries]
    out = Path(args.out or f"data/google_ad_news{'_ai' if args.ai else ''}.csv")
    print("Queries:")
    for query in queries:
        print(f"  {query}")
    print()

    articles = []
    for start, end in time_windows(args.years, args.window_days):
        found = []
        for query in queries:
            try:
                found.extend(fetch_window(query, start, end))
            except (requests.RequestException, ET.ParseError) as error:
                print(f"{start} -> {end}: FAILED ({error})")
            time.sleep(1)  # be polite to Google News
        print(f"{start} -> {end}: {len(found)} articles")
        articles.extend(found)

    if not articles:
        print("\nNo articles found. Nothing was saved.")
        return

    df = pd.DataFrame(articles)
    df = df.drop_duplicates(subset="link").drop_duplicates(subset=["title", "source"])
    df = df.sort_values("published").reset_index(drop=True)

    out.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out, index=False)

    print(f"\nSaved {len(df)} unique articles to {out}")
    print("\nArticles per year:")
    print(df["published"].dt.year.value_counts().sort_index().to_string())
    print("\nTop sources:")
    print(df["source"].value_counts().head(10).to_string())

### Options of step 1

In [ ]:
import shlex  # cuts the OPTIONS text into words, the way the command line does

# The options, written as on the command line. "" means every default.
# "--help" lists all of them (and then stops the cell, which is normal).
# Examples from the script:
#   OPTIONS = '--ai'
#   OPTIONS = '--years 2 --query \'intitle:Google "search market share"\''
OPTIONS = ""

### Run step 1

> **Warning:** Goes to the web and **replaces** `data/google_ad_news.csv` with a new collection: the results will not be the same as the ones in the assignment note. Skip this cell to keep the saved data.

In [ ]:
main()

## Step 2: `fetch_summaries.py`

Fetch the real address and the sub-header of the articles collected by google_ad_news.py.

Google News gives a redirect link and no summary. For every article this
script finds the publisher's own address, opens the page and reads the short
description the publisher wrote for it (the text shown under the headline in
search results and link previews).

The results go to their own CSV file, one row per link. The script can be
stopped and started again: it only fetches the links that are not done yet.
Some publishers block scripts or sit behind a paywall, so not every article
gets a description.

### Imports

In [ ]:
import argparse
import json
import re
import time
from concurrent.futures import ThreadPoolExecutor
from html.parser import HTMLParser
from pathlib import Path
from urllib.parse import quote, urlparse

import pandas as pd
import requests

### Constants

In [ ]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0 Safari/537.36"
}
# From Europe, Google News answers with a cookie-consent page instead of the
# article. This cookie says the choice was already made.
COOKIES = {"SOCS": "CAISHAgBEhJnd3NfMjAyMzA4MTAtMF9SQzIaAmVuIAEaBgiAo_CmBg"}
DECODE_URL = "https://news.google.com/_/DotsSplashUi/data/batchexecute"
# Where publishers put the description, best first
DESCRIPTION_TAGS = ["og:description", "description", "twitter:description"]

### Functions

In [ ]:
class MetaTags(HTMLParser):
    """Collect the <meta> tags of a page as {name: content}."""

    def __init__(self):
        super().__init__()
        self.found = {}

    def handle_starttag(self, tag, attrs):
        if tag != "meta":
            return
        attrs = dict(attrs)
        name = (attrs.get("property") or attrs.get("name") or "").lower()
        content = " ".join((attrs.get("content") or "").split())
        if name and content and name not in self.found:
            self.found[name] = content

In [ ]:
def real_url(link):
    """Turn a Google News redirect link into the publisher's address."""
    article_id = urlparse(link).path.split("/")[-1]
    page = requests.get(
        f"https://news.google.com/rss/articles/{article_id}", headers=HEADERS, cookies=COOKIES, timeout=20
    )
    page.raise_for_status()
    # The page carries a signature and a timestamp that Google asks for
    # before it gives the address away
    signature = re.search(r'data-n-a-sg="([^"]+)"', page.text)
    timestamp = re.search(r'data-n-a-ts="([^"]+)"', page.text)
    if not (signature and timestamp):
        raise ValueError("no signature on the Google News page")

    request = (
        '["garturlreq",[["X","X",["X","X"],null,null,1,1,"US:en",null,1,null,null,null,null,null,0,1],'
        f'"X","X",1,[1,1,1],1,1,null,0,0,null,0],"{article_id}",{timestamp.group(1)},"{signature.group(1)}"]'
    )
    answer = requests.post(
        DECODE_URL,
        headers={**HEADERS, "Content-Type": "application/x-www-form-urlencoded;charset=UTF-8"},
        cookies=COOKIES,
        data=f"f.req={quote(json.dumps([[['Fbv4je', request]]]))}",
        timeout=20,
    )
    answer.raise_for_status()
    return json.loads(json.loads(answer.text.split("\n\n")[1])[0][2])[1]

In [ ]:
def page_description(url):
    """Open an article and return the description from its <meta> tags ("" if none)."""
    page = requests.get(url, headers=HEADERS, timeout=20)
    page.raise_for_status()
    if "charset" not in page.headers.get("Content-Type", "").lower():
        page.encoding = "utf-8"
    tags = MetaTags()
    tags.feed(page.text)
    for name in DESCRIPTION_TAGS:
        if name in tags.found:
            return tags.found[name]
    return ""

In [ ]:
def fetch(link):
    """Return one result row for a link. Never raises: the problem goes in "status"."""
    row = {"link": link, "url": "", "description": "", "status": "ok"}
    try:
        row["url"] = real_url(link)
    except (requests.RequestException, ValueError, IndexError, KeyError, TypeError) as error:
        row["status"] = f"link not resolved ({type(error).__name__})"
        return row
    try:
        row["description"] = page_description(row["url"])
        if not row["description"]:
            row["status"] = "no description on the page"
    except requests.HTTPError as error:
        row["status"] = f"page blocked (HTTP {error.response.status_code})"
    except requests.RequestException as error:
        row["status"] = f"page not loaded ({type(error).__name__})"
    time.sleep(1)  # be polite
    return row

### main(): the steps, in the order they run

In [ ]:
def main():
    parser = argparse.ArgumentParser(description='Fetch the real address and the sub-header of the articles collected by google_ad_news.py.')
    parser.add_argument("--in", dest="source", default="data/google_ad_news.csv", help="CSV file with a link column")
    parser.add_argument("--out", default="data/article_summaries.csv", help="where the results are saved")
    parser.add_argument("--retry", action="store_true", help="try again the links that failed last time")
    parser.add_argument("--workers", type=int, default=4, help="how many articles to fetch at the same time (default: 4)")
    args = parser.parse_args(shlex.split(OPTIONS))

    links = pd.read_csv(args.source)["link"].drop_duplicates().tolist()
    out = Path(args.out)
    done = pd.read_csv(out, keep_default_na=False) if out.exists() else pd.DataFrame(columns=["link", "url", "description", "status"])
    if args.retry:
        done = done[done["status"] == "ok"]
    todo = [link for link in links if link not in set(done["link"])]
    print(f"{len(links)} links in {args.source}: {len(links) - len(todo)} already done, {len(todo)} to fetch\n")

    rows = done.to_dict("records")
    with ThreadPoolExecutor(max_workers=args.workers) as pool:
        for count, row in enumerate(pool.map(fetch, todo), start=1):
            rows.append(row)
            if count % 25 == 0 or count == len(todo):
                pd.DataFrame(rows).to_csv(out, index=False)
                print(f"{count}/{len(todo)} fetched")

    results = pd.DataFrame(rows)
    results = results[results["link"].isin(links)]
    print(f"\nSaved {len(rows)} rows to {out}")
    print(f"\nOf the {len(links)} links in {args.source}:")
    print(results["status"].str.replace(r" \(.*", "", regex=True).value_counts().to_string())

### Options of step 2

In [ ]:
import shlex  # cuts the OPTIONS text into words, the way the command line does

# The options, written as on the command line. "" means every default.
# "--help" lists all of them (and then stops the cell, which is normal).
# Examples from the script:
#   OPTIONS = '--in data/google_ad_news_ai.csv'
#   OPTIONS = '--retry'
OPTIONS = ""

### Run step 2

> **Warning:** Goes to the web. Only fetches the links not yet in `data/article_summaries.csv`, so a second run on the same data does nothing new.

In [ ]:
main()

## Step 3: `clean_news.py`

Clean the Google News headlines collected by google_ad_news.py.

The collection script casts a wide net, so most of what it returns only
mentions Google in passing. This script keeps the articles about Google's
search business, Google's ad revenue, or the traffic Google search sends to
publishers, in three steps:

1. Drop sources that publish evergreen explainers, how-to guides or Google's
   own marketing instead of news.
2. Tag every headline with the topics and the noise categories it matches,
   and keep the ones with at least one topic and no noise.
3. Group the articles that report the same story and keep the first one of
   each group. Two outlets rarely word a story the same way, so articles are
   compared on the words they share, in the headline and in the sub-header
   fetched by fetch_summaries.py.

Nothing is thrown away silently: the kept articles and the rejected ones go to
two separate CSV files, each with the reason for the decision.

### Imports

In [ ]:
import argparse
import math
import re
from collections import Counter
from pathlib import Path

import pandas as pd

### Constants

In [ ]:
# Sources that are not news: explainer and statistics pages that get re-dated
# every time they are updated, how-to guides for advertisers, and Google's own
# product marketing.
BLOCKED_SOURCES = {
    "FourWeekMBA",
    "Statista",
    "Semrush",
    "Business Model Analyst",
    "Business Strategy Hub",
    "SmartAsset.com",
    "Shopify",
    "AFFiNCO",
    "SQ Magazine",
    "Amra & Elma",
    "Themeisle",
    "G2 Learning Hub",
    "Enterprise Apps Today",
    "Business.com",
    "blog.google",
    "Google",
    "Google Business Profile",
}

# What we are looking for. A headline needs at least one of these.
# (The search queries all use intitle:Google, so "Google" is already there.)
TOPICS = {
    "search_business": (
        r"\bsearch(es|ing)?\b|searchgpt|perplexity|ai mode|ai overviews?|market share|\bbing\b|safari"
        r"|dominance|google killer|open web|blue links"
    ),
    "ad_revenue": (
        r"\b(ad|ads|advertising) (revenue|sales|business|income|growth|market|spend\w*|dollars|empire|machine|crown|giant|seller|platform)"
        r"|(revenue|sales|billions?|profits?) (from|in|on) (search )?(ads?|advertising)\b"
        r"|search (ads?|advertising)\b|digital advertising|chatgpt ads"
        r"|\bads? (in|into|inside|on|to) (ai overviews?|ai mode|gemini|search|chatbot)"
    ),
    "earnings": (
        r"\b(earnings|revenues?|quarterly|q[1-4]|profits?)\b"
        r"|\$[\d.]+ ?(b|bn|billion|t|trillion)\b.*\b(quarter|sales|year)\b"
    ),
    "publisher_traffic": (
        r"\btraffic\b|\breferrals?\b|zero.click|google zero|click.through|clickthrough"
        r"|\bpublishers?\b|\bnewspapers?\b|\bdailies\b|\bnewsrooms?\b|\bjournalis\w+|\bwebsites?\b"
        r"|news (sites?|industry|media|outlets?|publishers?|scraping)|media (companies|outlets|groups?|briefing)"
        r"|ai (summar\w+|answers?)|independent sites|google discover|\bcrawlers?\b|(french|german|local) media"
    ),
}

# What we are not looking for. Any of these rejects the headline, even when a
# topic matched.
NOISE = {
    "howto": (
        r"^how to\b|: how to\b|\bguide\b|\btips\b|\bstatistics\b|\bstats\b|\btutorial\b|\bchecklist\b|\bcourses?\b"
        r"|\bswot\b|\bsalary\b|marketing strategy|\bbest practices\b|\broundup\b|\beasy steps\b"
        r"|\b\d+ (best )?(ways|tips|tools|facts|examples|software|services)\b|\bvs\.? .*comparison"
    ),
    "ads_product": (
        r"pmax|performance max|demand gen|enhanced conversions|asset groups?|smart bidding|merchant center"
        r"|google ads (adds|rolls|simplifies|launches|tests|updates|introduces|now|is rolling|expands|gets|to|beta|api|editor|bug)"
        r"|keyword planner|ad strength|responsive search ads|broad match|negative keywords|local services ads"
        r"|budget pacing|bulk ad|campaign.level|\bai max\b|search themes|ads agency|target roas"
    ),
    "seo_chatter": (
        r"ranking (volatility|update)|search console|search liaison|helpful content"
        r"|google business profiles?|\bseo (tips|tools|strategy|checklist)\b|structured data"
    ),
    "stock_pick": (
        r"should you buy|stock forecast|price target|\bbuy the (shares|dip|stock)\b|millionaire.maker"
        r"|\bstocks? to (buy|watch|own)\b|\bbuy or sell\b|better buy|\bis \w+ a buy\b"
    ),
    "other_product": (
        r"\bpixel\b|google (maps|play|photos|workspace|drive|wallet|tv|home|fi|flights|classroom|shopping)"
        r"|\bwaymo\b|\byoutube (tv|music|premium|shorts)\b|\bchromebooks?\b|\bnest\b"
        # Google Cloud on its own; earnings stories about the whole company stay in
        r"|^(?!.*(alphabet|search|\bads?\b|advertising|earnings|\bq[1-4]\b)).*google cloud"
    ),
}

# How much two articles must overlap to count as the same story (see find_same_stories)
SIMILARITY = 0.25

# Words that say nothing about which story a headline tells. Every headline
# names Google, so "google" is one of them.
STOP_WORDS = {
    "a", "an", "the", "to", "of", "in", "on", "for", "and", "as", "is", "its", "s", "it", "at", "by", "with",
    "google", "googles", "alphabet", "says", "say", "said", "new", "from", "that", "this", "how", "why", "what",
    "are", "be", "will", "could", "may", "after", "over", "amid", "into", "than", "more", "has", "have", "was",
    "were", "their", "they", "but", "not", "you", "your", "can", "about", "which", "who",
}

### Functions

In [ ]:
def matching_labels(title, patterns):
    """Return the comma-separated names of the patterns found in the headline."""
    title = str(title).lower().replace("’", "'")
    return ", ".join(name for name, pattern in patterns.items() if re.search(pattern, title))

In [ ]:
def story_words(text):
    """Return the set of words and numbers of a text that can identify a story."""
    text = str(text).lower().replace("’", "'").replace("'s", "")
    words = set()
    for word in re.findall(r"[a-z]+|\$?\d+(?:\.\d+)?%?", text):
        if word in STOP_WORDS:
            continue
        # Crude stemming, so that "publishers" and "publisher" count as one word
        if word.isalpha() and len(word) > 4:
            word = re.sub(r"(ing|ers|er|ed|es|s)$", "", word)
        words.add(word)
    return words

In [ ]:
def find_same_stories(df, similarity, window_days):
    """Return, for each row, the position of the first article of its story.

    A row that starts a story (or is alone) gets its own position. df must be
    sorted by date.

    Every article is turned into a list of weighted words: a word counts more
    when few articles use it ("penske" says more than "search"), and headline
    words count double. Two articles tell the same story when they were
    published within `window_days` of each other and their weighted words
    overlap by at least `similarity` (cosine similarity: 0 = nothing in
    common, 1 = same words).
    """
    counts = []
    for title, description in zip(df["title"], df["description"]):
        in_title = story_words(title)
        counts.append(Counter(in_title) + Counter(in_title | story_words(description)))
    articles_using = Counter(word for count in counts for word in count)

    vectors = []
    for count in counts:
        vector = {word: n * math.log(len(counts) / articles_using[word]) for word, n in count.items()}
        length = math.sqrt(sum(weight**2 for weight in vector.values())) or 1
        vectors.append({word: weight / length for word, weight in vector.items()})

    dates = df["published"].tolist()
    window = pd.Timedelta(days=window_days)
    first_of_story = list(range(len(df)))
    for i in range(len(df)):
        best = similarity
        for j in range(i - 1, -1, -1):
            if dates[i] - dates[j] > window:
                break
            alike = sum(weight * vectors[j].get(word, 0) for word, weight in vectors[i].items())
            if alike >= best:
                best = alike
                first_of_story[i] = first_of_story[j]
    return first_of_story

### Constants

In [ ]:
# A description the publisher wrote counts fully. The ones written afterwards for
# blocked pages count less: they are a second-hand reading of the article, and the
# ones made from a search-result snippet never saw the article at all.
DESCRIPTION_WEIGHTS = {"publisher": 1.0, "written_from_text": 0.5, "written_from_snippet": 0.25}

### Functions

In [ ]:
def description_source(status):
    """Turn the status column of the summaries file into who wrote the description."""
    if status.startswith("summary written from search-result snippet"):
        return "written_from_snippet"
    if status.startswith("summary written from article text"):
        return "written_from_text"
    return "publisher"

### main(): the steps, in the order they run

In [ ]:
def main():
    parser = argparse.ArgumentParser(description='Clean the Google News headlines collected by google_ad_news.py.')
    parser.add_argument("--in", dest="source", default="data/google_ad_news.csv", help="CSV file to clean")
    parser.add_argument("--summaries", default="data/article_summaries.csv", help="CSV file made by fetch_summaries.py (default: data/article_summaries.csv)")
    parser.add_argument("--leads", default="data/article_leads.csv", help="CSV file with the opening paragraphs of the articles (default: data/article_leads.csv)")
    parser.add_argument("--similarity", type=float, default=SIMILARITY, help=f"how much two articles must overlap to count as the same story, 0-1 (default: {SIMILARITY})")
    parser.add_argument("--window-days", type=int, default=4, help="only compare articles published this many days apart (default: 4)")
    args = parser.parse_args(shlex.split(OPTIONS))

    source = Path(args.source)
    df = pd.read_csv(source)
    df["published"] = pd.to_datetime(df["published"], utc=True)
    df = df.sort_values("published").reset_index(drop=True)
    print(f"Loaded {len(df)} articles from {source}")

    # Add the real address and the sub-header of each article, when we have them
    if Path(args.summaries).exists():
        summaries = pd.read_csv(args.summaries, keep_default_na=False)[["link", "url", "description", "status"]]
        df = df.merge(summaries.drop_duplicates(subset="link"), on="link", how="left")
    else:
        print(f"{args.summaries} not found: run fetch_summaries.py first to compare sub-headers too")
        df["url"] = df["description"] = df["status"] = ""
    df[["url", "description", "status"]] = df[["url", "description", "status"]].fillna("")
    # A text found under several different headlines describes the site, not the article
    site_text = df.groupby("description")["title"].transform("nunique") > 1
    df.loc[site_text, "description"] = ""
    # Who wrote the description, and how much it should count in the analysis
    df["description_source"] = df["status"].map(description_source).where(df["description"] != "", "")
    df["description_weight"] = df["description_source"].map(DESCRIPTION_WEIGHTS).fillna(0.0)
    df = df.drop(columns="status")
    # The opening paragraphs of each article, when they could be read
    if Path(args.leads).exists():
        leads = pd.read_csv(args.leads, keep_default_na=False)[["link", "lead"]]
        df = df.merge(leads.drop_duplicates(subset="link"), on="link", how="left")
    else:
        df["lead"] = ""
    df["lead"] = df["lead"].fillna("")
    print(f"{(df['description'] != '').sum()} of them have a sub-header\n")

    df["topics"] = df["title"].map(lambda title: matching_labels(title, TOPICS))
    df["noise"] = df["title"].map(lambda title: matching_labels(title, NOISE))
    df["reason"] = ""
    df["story_size"] = 1

    # Step 1: blocked sources
    blocked = df["source"].isin(BLOCKED_SOURCES)
    df.loc[blocked, "reason"] = "blocked source"

    # Step 2: topic and noise patterns
    remaining = df["reason"] == ""
    df.loc[remaining & (df["noise"] != ""), "reason"] = "noise: " + df["noise"]
    df.loc[remaining & (df["noise"] == "") & (df["topics"] == ""), "reason"] = "no topic matched"

    # Step 3: same story, among the articles that are still in
    remaining = df[df["reason"] == ""]
    first_of_story = remaining.index[find_same_stories(remaining, args.similarity, args.window_days)]
    first_of_story = pd.Series(first_of_story, index=remaining.index)
    df.loc[remaining.index, "story_size"] = first_of_story.map(first_of_story.value_counts())
    repeats = first_of_story[first_of_story != first_of_story.index]
    df.loc[repeats.index, "reason"] = "same story as: " + df.loc[repeats, "title"].values

    kept = df[df["reason"] == ""].drop(columns=["noise", "reason"])
    rejected = df[df["reason"] != ""]

    kept_path = source.with_name(f"{source.stem}_clean.csv")
    rejected_path = source.with_name(f"{source.stem}_rejected.csv")
    kept.to_csv(kept_path, index=False)
    rejected.to_csv(rejected_path, index=False)

    print("Rejected:")
    reasons = rejected["reason"].str.replace(r"^same story as: .*", "same story as an earlier article", regex=True)
    print(reasons.value_counts().to_string())
    print(f"\nKept {len(kept)} of {len(df)} articles -> {kept_path}")
    print(f"Rejected {len(rejected)} articles -> {rejected_path}")
    print(f"\nStories covered by several articles: {(kept['story_size'] > 1).sum()}. The biggest:")
    print(kept.nlargest(5, "story_size")[["story_size", "title"]].to_string(index=False, header=False))
    print("\nKept articles per topic (an article can have several):")
    print(kept["topics"].str.split(", ").explode().value_counts().to_string())
    print("\nKept articles per year:")
    print(kept["published"].dt.year.value_counts().sort_index().to_string())

### Options of step 3

In [ ]:
import shlex  # cuts the OPTIONS text into words, the way the command line does

# The options, written as on the command line. "" means every default.
# "--help" lists all of them (and then stops the cell, which is normal).
# Examples from the script:
#   OPTIONS = '--in data/google_ad_news_ai.csv'
#   OPTIONS = '--similarity 0.3'
OPTIONS = ""

### Run step 3

In [ ]:
main()

## Step 4: `fetch_leads.py`

Fetch the opening paragraphs of the articles kept by clean_news.py.

The sub-header of an article is often a few words. This script opens every
article, reads its paragraphs and keeps the first ones, word for word: the
"lead" of the article. When the publisher blocks the script, it tries the copy
kept by the Internet Archive.

The results go to their own CSV file, one row per link. The script can be
stopped and started again: it only fetches the links that are not done yet.
Paywalls and bot protection mean that not every article gets a lead.

### Imports

In [ ]:
import argparse
import re
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from html.parser import HTMLParser
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import requests

### Constants

In [ ]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0 Safari/537.36"
}
# "2id_" asks the Internet Archive for its latest copy of a page, as it was saved
ARCHIVE_URL = "https://web.archive.org/web/2id_/{url}"
# Parts of a page that are never the article
SKIPPED_TAGS = {"script", "style", "noscript", "nav", "header", "footer", "aside", "form", "figure", "figcaption", "button"}
# Words too common to tell if a paragraph is about the headline
FILLER = set(
    "the a an and or of to in on for with from by at as is are was were be been its it this that than how why what "
    "when will can could may not new says say said more over after about into your you our their has have had but "
    "his her they them who which google alphabet".split()
)
# Paragraphs that belong to the page, not to the article
PAGE_FURNITURE = re.compile(
    r"subscribe|sign up|sign in|log in|newsletter|cookie|^advertisement|follow us|all rights reserved|click here"
    r"|read more|getty images|listen to this|min read|share this|©|free account|javascript|ad-?blocker|keep reading"
    r"|unlock|already a (member|subscriber)|terms of (use|service)|privacy policy|disclosure|editor.s note"
    r"|opinions expressed|purchase through links|earn a (small )?commission|affiliate|screenshot|AP Photo"
    r"|via Getty|round-up of|^hello from|confirmation link|spam folder|^by ",
    re.I,
)
AUTHOR_BIOGRAPHY = re.compile(
    r"^.{0,80}\b(is|was) (an?|the) .{0,80}\b(writer|reporter|editor|journalist|correspondent|columnist|contributor|analyst|author|founder|professor)\b",
    re.I,
)
MIN_WORDS, ENOUGH_WORDS, MAX_WORDS = 15, 60, 130
# The Internet Archive refuses visitors that ask too fast: one request at a time, with a pause
ARCHIVE_TURN = threading.Lock()
ARCHIVE_PAUSE = 4
# Set when the Internet Archive keeps refusing: the script stops asking it until the next run
ARCHIVE_REFUSING = threading.Event()

### Functions

In [ ]:
class Paragraphs(HTMLParser):
    """Collect the text of the <p> tags of a page, and the name of its author."""

    def __init__(self):
        super().__init__()
        self.found = []
        self.author = ""
        self.skipping = 0  # how many skipped tags we are inside
        self.current = None  # the pieces of the paragraph being read

    def handle_starttag(self, tag, attrs):
        if tag in SKIPPED_TAGS:
            self.skipping += 1
        elif tag == "p" and not self.skipping:
            self.current = []
        elif tag == "meta":
            attrs = dict(attrs)
            if (attrs.get("name") or attrs.get("property") or "").lower() in ("author", "article:author") and not self.author:
                self.author = " ".join((attrs.get("content") or "").split())

    def handle_endtag(self, tag):
        if tag in SKIPPED_TAGS:
            self.skipping = max(0, self.skipping - 1)
        elif tag == "p" and self.current is not None:
            text = " ".join("".join(self.current).split())
            if text:
                self.found.append(text)
            self.current = None

    def handle_data(self, data):
        if self.current is not None and not self.skipping:
            self.current.append(data)

In [ ]:
def story_words(text):
    """The words of a text that say what it is about."""
    return {word for word in re.findall(r"[a-z0-9$%']+", text.lower()) if len(word) > 2 and word not in FILLER}

In [ ]:
def lead_of(paragraphs, author, title):
    """Return the opening paragraphs of an article ("" if they are not about the headline)."""
    first_name = author.split()[0] if author else ""
    article = []
    for text in paragraphs:
        words = len(text.split())
        if words < MIN_WORDS or PAGE_FURNITURE.search(text) or AUTHOR_BIOGRAPHY.search(text):
            continue
        if author and (text.startswith(author) or (len(first_name) > 3 and re.match(re.escape(first_name) + "['’]s ", text))):
            continue
        if words < 25 and not re.search(r"[.!?…”\"]$", text):  # a caption or a menu, not a sentence
            continue
        article.append(text)

    # Start with the opening when it is about the headline; if not, with the
    # first of the next paragraphs that is
    headline = story_words(title)
    if len(story_words(" ".join(article[:2])) & headline) >= 2:
        start = 0
    else:
        start = next((i for i, text in enumerate(article[:6]) if len(story_words(text) & headline) >= 2), None)
        if start is None:
            return ""

    lead = []
    for text in article[start : start + 5]:
        if lead and (text[:50] == lead[-1][:50] or text in " ".join(lead)):  # the page repeats its first paragraph
            if len(text) > len(lead[-1]) and text[:50] == lead[-1][:50]:
                lead[-1] = text
            continue
        lead.append(text)
        if sum(len(part.split()) for part in lead) >= ENOUGH_WORDS:
            break
    lead = " ".join(lead)
    if len(lead.split()) > MAX_WORDS:  # cut at the end of a sentence
        cut = " ".join(lead.split()[:MAX_WORDS])
        end = max(cut.rfind(". "), cut.rfind(".” "), cut.rfind("? "))
        lead = cut[: end + 1] if end > 200 else cut + "…"
    return lead

In [ ]:
def open_page(address):
    """Open a page. Archive pages wait for their turn, and try once more when told to slow down."""
    if "web.archive.org" not in address:
        return requests.get(address, headers=HEADERS, timeout=30)
    with ARCHIVE_TURN:
        page = requests.get(address, headers=HEADERS, timeout=60)
        if page.status_code == 429:
            time.sleep(30)
            page = requests.get(address, headers=HEADERS, timeout=60)
            if page.status_code == 429:
                ARCHIVE_REFUSING.set()
        time.sleep(ARCHIVE_PAUSE)
        return page

In [ ]:
def page_lead(address, title, url):
    """Open a page and return the lead of the article in it ("" if none)."""
    page = open_page(address)
    page.raise_for_status()
    # A link that lands on the home page did not give the article
    landed = page.url.split("id_/", 1)[-1] if "web.archive.org" in page.url else page.url
    if urlparse(landed).path.strip("/") == "" and urlparse(url).path.strip("/") != "":
        return ""
    if "charset" not in page.headers.get("Content-Type", "").lower():
        page.encoding = "utf-8"
    parser = Paragraphs()
    parser.feed(page.text)
    return lead_of(parser.found, parser.author.split(",")[0].strip(), title)

In [ ]:
def fetch(article):
    """Return one result row for an article. Never raises: the problem goes in "status"."""
    row = {"link": article["link"], "url": article["url"], "lead": "", "lead_source": "", "status": ""}
    places = [("page", article["url"]), ("archive", ARCHIVE_URL.format(url=article["url"]))]
    for source, address in places:
        if source == "archive" and ARCHIVE_REFUSING.is_set():
            row["status"] += "; archive not asked (it was refusing requests)"
            break
        try:
            row["lead"] = page_lead(address, article["title"], article["url"])
            problem = "" if row["lead"] else "no paragraph about the headline"
        except requests.HTTPError as error:
            problem = f"blocked (HTTP {error.response.status_code})"
        except requests.RequestException as error:
            problem = f"not loaded ({type(error).__name__})"
        if row["lead"]:
            row["lead_source"], row["status"] = source, "ok"
            break
        row["status"] = f"{row['status']}; {source} {problem}".strip("; ")
        time.sleep(1)  # be polite
    return row

### main(): the steps, in the order they run

In [ ]:
def main():
    parser = argparse.ArgumentParser(description='Fetch the opening paragraphs of the articles kept by clean_news.py.')
    parser.add_argument("--in", dest="source", default="data/google_ad_news_clean.csv", help="CSV file with link, url and title columns")
    parser.add_argument("--out", default="data/article_leads.csv", help="where the results are saved")
    parser.add_argument("--retry", action="store_true", help="try again the links that failed last time")
    parser.add_argument("--workers", type=int, default=4, help="how many articles to fetch at the same time (default: 4)")
    args = parser.parse_args(shlex.split(OPTIONS))

    articles = pd.read_csv(args.source, keep_default_na=False).drop_duplicates(subset="link")
    out = Path(args.out)
    columns = ["link", "url", "lead", "lead_source", "status"]
    done = pd.read_csv(out, keep_default_na=False) if out.exists() else pd.DataFrame(columns=columns)
    if args.retry:
        done = done[done["lead"] != ""]
    todo = articles[~articles["link"].isin(set(done["link"]))][["link", "url", "title"]].to_dict("records")
    print(f"{len(articles)} articles in {args.source}: {len(articles) - len(todo)} already done, {len(todo)} to fetch\n")

    rows = done.to_dict("records")
    with ThreadPoolExecutor(max_workers=args.workers) as pool:
        for count, row in enumerate(pool.map(fetch, todo), start=1):
            rows.append(row)
            if count % 25 == 0 or count == len(todo):
                pd.DataFrame(rows, columns=columns).to_csv(out, index=False)
                print(f"{count}/{len(todo)} fetched")

    results = pd.DataFrame(rows, columns=columns)
    results = results[results["link"].isin(articles["link"])]
    print(f"\nSaved {len(rows)} rows to {out}")
    print(f"\nOf the {len(articles)} articles in {args.source}:")
    print(results["lead_source"].replace("", "no lead").value_counts().to_string())
    if ARCHIVE_REFUSING.is_set():
        print("\nThe Internet Archive refused requests: run again later with --retry to try the missing ones there")

### Options of step 4

In [ ]:
import shlex  # cuts the OPTIONS text into words, the way the command line does

# The options, written as on the command line. "" means every default.
# "--help" lists all of them (and then stops the cell, which is normal).
# Examples from the script:
#   OPTIONS = '--in data/google_ad_news_ai_clean.csv'
#   OPTIONS = '--retry'
OPTIONS = ""

### Run step 4

> **Warning:** Goes to the web. By default it writes to `data/article_leads.csv`, the file that was made by hand with TinyFish: use `OPTIONS = '--out data/article_leads_script.csv'` to keep it.

In [ ]:
main()

## Step 5: `analyze_news.py`

Check the cleaned headlines against the problem statement.

Problem statement: usage of Google's legacy search is down, maybe because of
AI, and so Google's ad revenue is down. That is two claims, and every article
is checked against each of them:

- search volume is down
- ad revenue is down

The rule is the same for both and uses general vocabulary only, no phrase taken
from a particular headline. The headline and the sub-header are cut into
clauses. A clause says something about a claim when it names the subject
("searches", "ad revenue"...) and has a direction word ("falls", "grows"...)
a few words away:

    subject + down word  -> support   (the claim is true)
    subject + up word    -> oppose    (the claim is false)
    a "not" just before the direction word turns it around

Stance of an article on one claim:

    support    only clauses that support it
    oppose     only clauses that oppose it
    both       clauses on each side
    no claim   the subject is named, but with no direction
    (empty)    the subject is not named

An article can be about both subjects: it then has a stance on each claim, and
is counted in both. The `category` column puts the two claims together, and is
`others` when the article names neither subject.

Everything we have on an article is read: the headline, the sub-header and the
opening paragraphs saved in data/article_leads.csv. --text reads only a part of
it, to compare.

### Imports

In [ ]:
import argparse
import re
from pathlib import Path

import pandas as pd

### Constants

In [ ]:
# How an article names each subject
SUBJECTS = {
    "search_volume": (
        r"\bsearches\b|\bqueries\b|\bsearching\b"
        r"|\bsearch (volumes?|usage|activity|queries|market share|share)\b"
    ),
    "ad_revenue": (
        r"\b(ads?|advertising|search|network) (ads? |advertising )?(revenues?|sales|income)\b"
        r"|\bsearch (ads?|advertising)\b|\brevenues? from (ads?|advertising|search)\b"
    ),
}

# Direction words, as word beginnings ("declin" finds declines, declined, declining)
DOWN = (
    "fall", "fell", "drop", "declin", "decreas", "plung", "plummet", "slip", "slid", "sink", "sank", "shrink",
    "shrank", "lose", "losing", "loss", "lost", "dip", "erod", "tumbl", "slump", "fading", "fades", "weak",
    "collaps", "crash",
)
UP = (
    "rise", "rising", "rose", "grow", "grew", "increas", "surg", "soar", "jump", "climb", "gain", "record",
    "boost", "lift", "beat", "strong", "accelerat",
)
# "more queries", "less searching": only a direction when right next to the subject
COMPARATIVES = {"more": "up", "higher": "up", "less": "down", "fewer": "down", "lower": "down"}
# "up" and "down" are in many expressions ("teams up", "drying up"). They only count
# as "is up", "was down"... or in front of a figure ("up 11%", "down nearly 20%")
UP_DOWN = r"\b(?:is|are|was|were) (up|down)\b|\b(up|down) (?:by |again |nearly |about |almost |over |roughly )?\$?\d"
NEGATIONS = {"not", "no", "never", "without", "isn't", "aren't", "doesn't", "don't", "didn't", "hasn't", "haven't", "won't"}

# A clause about one of these, which does not name Google, gives their figures
OTHER_COMPANIES = r"\b(meta|facebook|microsoft|bing|amazon|reddit|openai|chatgpt|perplexity|pinterest|snap|tiktok|apple)\b"
GOOGLE = r"\b(google|alphabet)\b"
# A clause with one of these is about what websites get, not about Google: their
# ad revenue, or the searches that send them no visitor
WEBSITE_SIDE = r"\b(publishers?|websites?|sites?|creators?|bloggers?|zero.click)\b"

# Where a sentence is cut into clauses. A dot only counts when a space follows it, to keep "10.4%" whole.
CLAUSE_BREAK = r"[.!?;:|]\s+|,\s+|\s[-–—&]\s|\b(?:while|but|as|despite|although|though|amid|after)\b"

### Functions

In [ ]:
def clean_text(text):
    """Lower-case a text and replace the curly apostrophes, to make it easy to match."""
    return str(text).lower().replace("’", "'").replace("‘", "'")

In [ ]:
def direction_of(word, distance, clause):
    """Return 'down', 'up' or '' for a word found `distance` words from the subject."""
    text = word.group()
    if text in COMPARATIVES:
        return COMPARATIVES[text] if distance == 1 else ""
    if text in ("up", "down"):
        counts = any(match.start(1 if match.group(1) else 2) == word.start() for match in re.finditer(UP_DOWN, clause))
        return text if counts else ""
    if text.startswith(DOWN):
        return "down"
    if text.startswith(UP):
        return "up"
    return ""

In [ ]:
def read_clause(clause, subject_pattern, window):
    """Return what a clause says about a subject: 'support', 'oppose', 'named' or ''.

    'named' means the subject is there with no direction word within `window`
    words of it.
    """
    subject = re.search(subject_pattern, clause)
    if not subject:
        return ""
    if re.search(OTHER_COMPANIES, clause) and not re.search(GOOGLE, clause):
        return ""
    if re.search(WEBSITE_SIDE, clause):
        return ""

    words = list(re.finditer(r"[a-z']+", clause))
    in_subject = [i for i, word in enumerate(words) if word.start() < subject.end() and word.end() > subject.start()]
    # Closest direction word first
    nearby = sorted(
        (min(abs(i - j) for j in in_subject), i)
        for i in range(len(words))
        if i not in in_subject and min(abs(i - j) for j in in_subject) <= window
    )
    for distance, i in nearby:
        direction = direction_of(words[i], distance, clause)
        if not direction:
            continue
        if any(word.group() in NEGATIONS for word in words[max(0, i - 3):i]):
            direction = "up" if direction == "down" else "down"
        return "support" if direction == "down" else "oppose"
    return "named"

In [ ]:
def stance_on(text, subject_pattern, window):
    """Return the stance of a text on one claim, and the clauses it was read from."""
    found = {}
    for clause in re.split(CLAUSE_BREAK, text):
        reading = read_clause(clause, subject_pattern, window)
        if reading:
            found.setdefault(reading, []).append(clause.strip())

    if "support" in found and "oppose" in found:
        stance = "both"
    elif "support" in found:
        stance = "support"
    elif "oppose" in found:
        stance = "oppose"
    elif "named" in found:
        stance = "no claim"
    else:
        stance = ""
    evidence = " | ".join(f"{reading}: {clause}" for reading in ("support", "oppose") for clause in found.get(reading, []))
    return stance, evidence

In [ ]:
def overall(stances):
    """Put the stances on the two claims together in one category."""
    stances = set(stances)
    if "both" in stances or {"support", "oppose"} <= stances:
        return "both"
    if "support" in stances:
        return "support"
    if "oppose" in stances:
        return "oppose"
    if "no claim" in stances:
        return "no claim"
    return "others"

### Constants

In [ ]:
# What --text reads. Every sub-header is read, the ones written by Claude too:
# the `description_source` column says which ones they are.
TEXTS = {"all": ["title", "description", "lead"], "header": ["title", "description"], "lead": ["lead"]}

### Functions

In [ ]:
def classify(article, window, parts):
    """Return the stance of an article on each claim and its overall category."""
    text = " . ".join(clean_text(article[part]) for part in parts)

    result = {}
    for name, pattern in SUBJECTS.items():
        result[f"{name}_down"], result[f"{name}_evidence"] = stance_on(text, pattern, window)
        # One yes/no column per subject: an article can be about both
        result[f"about_{name}"] = "yes" if result[f"{name}_down"] else "no"
    result["category"] = overall(result[f"{name}_down"] for name in SUBJECTS)
    return pd.Series(result)

### main(): the steps, in the order they run

In [ ]:
def main():
    parser = argparse.ArgumentParser(description='Check the cleaned headlines against the problem statement.')
    parser.add_argument("--in", dest="source", default="data/google_ad_news_clean.csv", help="cleaned CSV file to analyse")
    parser.add_argument("--out", default="data/matching_analysis_results.csv", help="CSV file to write")
    parser.add_argument("--text", choices=TEXTS, default="all", help="what to read: all = everything, header = headline and sub-header, lead = opening paragraphs (default: all)")
    parser.add_argument("--leads", default="data/article_leads.csv", help="CSV file with the opening paragraphs (default: data/article_leads.csv)")
    parser.add_argument("--window", type=int, default=6, help="how many words apart a subject and a direction word can be (default: 6)")
    args = parser.parse_args(shlex.split(OPTIONS))

    df = pd.read_csv(args.source, keep_default_na=False)
    print(f"Loaded {len(df)} articles from {args.source}\n")

    # The opening paragraphs come from their own file, matched on the Google News link
    leads = pd.read_csv(args.leads, keep_default_na=False)[["link", "lead"]].drop_duplicates(subset="link")
    df = df.drop(columns="lead", errors="ignore").merge(leads, on="link", how="left")
    df["lead"] = df["lead"].fillna("")
    if args.text != "header":
        print(f"{(df['lead'] != '').sum()} of them have opening paragraphs\n")

    df = pd.concat([df, df.apply(classify, axis=1, window=args.window, parts=TEXTS[args.text])], axis=1)
    columns = [
        "published", "category", "about_search_volume", "search_volume_down", "about_ad_revenue", "ad_revenue_down",
        "title", "description", "lead", "search_volume_evidence", "ad_revenue_evidence", "source", "url",
        "description_source", "description_weight", "topics", "story_size",
    ]
    Path(args.out).parent.mkdir(parents=True, exist_ok=True)
    df[columns].to_csv(args.out, index=False)

    order = ["support", "oppose", "both", "no claim", "others"]
    print("Articles per category:")
    print(df["category"].value_counts().reindex(order, fill_value=0).to_string())
    print("\nWhat the articles are about (an article can be about both):")
    about = df[[f"about_{name}" for name in SUBJECTS]].eq("yes")
    print(about.sum().rename(lambda column: column.replace("about_", "").replace("_", " ")).to_string())
    print(f"neither          {(~about.any(axis=1)).sum()}")
    for name in SUBJECTS:
        print(f"\n'{name.replace('_', ' ')} is down':")
        print(df[f"{name}_down"].replace("", "not named").value_counts().reindex(order[:4] + ["not named"], fill_value=0).to_string())
    print("\nCategory per year:")
    print(pd.crosstab(df["published"].str[:4], df["category"]).reindex(columns=order, fill_value=0).to_string())
    print(f"\nWrote {len(df)} articles -> {args.out}")

### Options of step 5

In [ ]:
import shlex  # cuts the OPTIONS text into words, the way the command line does

# The options, written as on the command line. "" means every default.
# "--help" lists all of them (and then stops the cell, which is normal).
# Examples from the script:
#   OPTIONS = '--text header --out data/matching_analysis_results_header.csv'
#   OPTIONS = '--text lead --out data/matching_analysis_results_leads.csv'
#   OPTIONS = '--window 4'
OPTIONS = ""

### Run step 5

In [ ]:
main()

## Step 6: `scan_topics.py`

Count the terms used in the articles that do not take a side on the problem statement.

analyze_news.py puts every article in a category. Most of them neither support
nor oppose the two claims ("others" and "no claim"). This script looks at what
those articles talk about, and writes three tables:

1. distinctive   terms used by a bigger share of the scanned articles than of
                 the articles that take a side
2. rising        share of the scanned articles using a term in each period,
                 sorted by the change from the first period to the last one.
                 The terms that go down are at the bottom of the same file.
3. weak signals  terms that are rare, were first seen recently and are used by
                 more than one source

Rules:

- A term is one word, or two words that follow each other in the text. Words
  are lower-cased, a final "'s" is removed, and the common English words in
  STOPWORDS are left out. "ads" and "ad" stay two different terms: nothing is
  done about plurals.
- A term is counted once per article, however many times the article uses it.
  All the figures are numbers of articles, not numbers of occurrences.
- A share is a percentage of the articles of a group or of a period.
- "Recently" is measured from the newest article of the input file, not from
  today's date, so the same input always gives the same output.
- The script only counts. It does not say which terms matter: --min-articles,
  --weak-max, --min-sources, --recent-months and --min-period-articles are
  judgement calls, to be changed from the command line.

### Imports

In [ ]:
import argparse
import re
from pathlib import Path

import pandas as pd

### Constants

In [ ]:
# Common English words that say nothing about the subject of an article
STOPWORDS = set(
    """
    a about above after again against all also am an and any are aren't as at be because been before being below
    between both but by can can't cannot could couldn't did didn't do does doesn't doing don't down during each
    even few for from further had hadn't has hasn't have haven't having he her here hers herself him himself his
    how however i if in into is isn't it its itself just like may me might more most much must my myself new no
    nor not now of off on once one only or other our ours ourselves out over own per said same say says she
    should shouldn't so some still such than that the their theirs them themselves then there these they this
    those through to too two under until up us very was wasn't we were weren't what when where which while who
    whom whose why will with won't would wouldn't yet you your yours yourself yourselves
    according across ago already among another around back based became become been come comes de el get gets
    going got la last less let made make makes making many mr ms next often read since take takes three told
    use used using via want way well went year years month months week weeks day days today
    """.split()
)

# A word starts with a letter and can hold figures, apostrophes and hyphens ("q3", "zero-click")
WORD = r"[a-z][a-z0-9'-]*"
# Two words only make a term when nothing of this is between them
CHUNK_BREAK = r"[.!?;:,|()\[\]\"“”]|\s[-–—]\s"

# What --text reads, the same choices as in analyze_news.py
TEXTS = {"all": ["title", "description", "lead"], "header": ["title", "description"], "lead": ["lead"]}

### Functions

In [ ]:
def clean_text(text):
    """Lower-case a text and replace the curly apostrophes, to make it easy to match."""
    return str(text).lower().replace("’", "'").replace("‘", "'")

In [ ]:
def terms_of(text, ignore):
    """Return the set of terms of a text: its words and its pairs of words."""
    terms = set()
    for chunk in re.split(CHUNK_BREAK, clean_text(text)):
        words = [word.removesuffix("'s").strip("'-") for word in re.findall(WORD, chunk)]
        keep = [len(word) >= 2 and word not in STOPWORDS and word not in ignore for word in words]
        for i, word in enumerate(words):
            if not keep[i]:
                continue
            terms.add(word)
            if i + 1 < len(words) and keep[i + 1]:
                terms.add(f"{word} {words[i + 1]}")
    return terms

In [ ]:
def period_of(date, length):
    """Return the period a date is in: '2025', '2025-H1' or '2025-Q3'."""
    if length == "half":
        return f"{date.year}-H{1 if date.month <= 6 else 2}"
    if length == "quarter":
        return f"{date.year}-Q{(date.month - 1) // 3 + 1}"
    return str(date.year)

In [ ]:
def term_table(articles, parts, ignore):
    """Return one row per article and term, with the columns of the article next to it."""
    text = articles[parts].agg(" . ".join, axis=1)
    terms = text.apply(lambda value: sorted(terms_of(value, ignore))).explode().dropna().rename("term")
    # The join repeats the row number of an article for each of its terms: number the rows again
    return articles.join(terms, how="inner").reset_index(drop=True)

In [ ]:
def distinctive(table, sizes, min_articles):
    """Compare the share of each term in the scanned articles and in the ones that take a side."""
    counts = pd.crosstab(table["term"], table["group"]).reindex(columns=["scan", "compare"], fill_value=0)
    counts = counts[counts["scan"] >= min_articles]
    result = pd.DataFrame({
        "articles_scan": counts["scan"],
        "share_scan": (100 * counts["scan"] / max(sizes["scan"], 1)).round(1),
        "articles_compare": counts["compare"],
        "share_compare": (100 * counts["compare"] / max(sizes["compare"], 1)).round(1),
    })
    result["difference"] = (result["share_scan"] - result["share_compare"]).round(1)
    # No ratio when the other group never uses the term: nothing to divide by
    result["ratio"] = (result["share_scan"] / result["share_compare"].where(result["share_compare"] > 0)).round(1)
    return result.reset_index().sort_values(["difference", "term"], ascending=[False, True], kind="stable")

In [ ]:
def rising(table, period_sizes, min_articles):
    """Give the share of each term per period and the change from the first period to the last."""
    periods = list(period_sizes.index)
    counts = pd.crosstab(table["term"], table["period"]).reindex(columns=periods, fill_value=0)
    counts = counts[counts.sum(axis=1) >= min_articles]
    result = (100 * counts / period_sizes).round(1).add_prefix("share_")
    result.insert(0, "articles", counts.sum(axis=1))
    result["change"] = (result[f"share_{periods[-1]}"] - result[f"share_{periods[0]}"]).round(1)
    return result.reset_index().sort_values(["change", "term"], ascending=[False, True], kind="stable")

In [ ]:
def weak_signals(table, cutoff, weak_min, weak_max, min_sources):
    """Keep the rare terms first seen after `cutoff` and used by enough sources."""
    table = table.sort_values(["published", "url"], kind="stable")
    result = table.groupby("term").agg(
        articles=("url", "size"),
        sources=("source", "nunique"),
        first_seen=("published", "min"),
        last_seen=("published", "max"),
        example_title=("title", "first"),
        example_url=("url", "first"),
    )
    result = result[
        result["articles"].between(weak_min, weak_max)
        & (result["sources"] >= min_sources)
        & (result["first_seen"] >= cutoff)
    ].reset_index()
    result = result.sort_values(["first_seen", "articles", "term"], ascending=[False, False, True], kind="stable")
    for column in ("first_seen", "last_seen"):
        result[column] = result[column].dt.strftime("%Y-%m-%d")
    return result

### main(): the steps, in the order they run

In [ ]:
def main():
    parser = argparse.ArgumentParser(description='Count the terms used in the articles that do not take a side on the problem statement.')
    parser.add_argument("--in", dest="source", default="data/matching_analysis_results.csv", help="CSV file written by analyze_news.py")
    parser.add_argument("--out", default="data/topic_scan", help="beginning of the names of the three CSV files to write (default: data/topic_scan)")
    parser.add_argument("--scan", default="others,no claim", help="categories to scan, separated by commas (default: others,no claim)")
    parser.add_argument("--compare", default="support,oppose,both", help="categories to compare them with (default: support,oppose,both)")
    parser.add_argument("--text", choices=TEXTS, default="all", help="what to read: all = everything, header = headline and sub-header, lead = opening paragraphs (default: all)")
    parser.add_argument("--ignore", default="", help="words to leave out on top of the common English words, separated by commas")
    parser.add_argument("--min-articles", type=int, default=5, help="tables 1 and 2: smallest number of scanned articles a term must be in (default: 5)")
    parser.add_argument("--period", choices=["year", "half", "quarter"], default="year", help="table 2: length of a period (default: year)")
    parser.add_argument("--min-period-articles", type=int, default=30, help="table 2: periods with fewer scanned articles are left out (default: 30)")
    parser.add_argument("--weak-min", type=int, default=2, help="table 3: smallest number of articles for a rare term (default: 2)")
    parser.add_argument("--weak-max", type=int, default=5, help="table 3: biggest number of articles for a rare term (default: 5)")
    parser.add_argument("--min-sources", type=int, default=2, help="table 3: smallest number of different sources (default: 2)")
    parser.add_argument("--recent-months", type=int, default=12, help="table 3: a term is recent when first seen in the last N months of the file (default: 12)")
    parser.add_argument("--show", type=int, default=15, help="how many rows of each table to print (default: 15)")
    args = parser.parse_args(shlex.split(OPTIONS))

    df = pd.read_csv(args.source, keep_default_na=False)
    df["published"] = pd.to_datetime(df["published"], utc=True)
    df["period"] = df["published"].apply(period_of, length=args.period)
    print(f"Loaded {len(df)} articles from {args.source}")

    scan = [name.strip() for name in args.scan.split(",") if name.strip()]
    compare = [name.strip() for name in args.compare.split(",") if name.strip()]
    df["group"] = ""
    df.loc[df["category"].isin(compare), "group"] = "compare"
    df.loc[df["category"].isin(scan), "group"] = "scan"
    sizes = df["group"].value_counts()
    sizes = {group: int(sizes.get(group, 0)) for group in ("scan", "compare")}
    print(f"Scanned: {sizes['scan']} articles ({', '.join(scan)})")
    print(f"Compared with: {sizes['compare']} articles ({', '.join(compare)})")
    if not sizes["scan"]:
        raise SystemExit("No article to scan: check --scan against the `category` column.")

    ignore = {word.strip().lower() for word in args.ignore.split(",") if word.strip()}
    table = term_table(df[df["group"] != ""], TEXTS[args.text], ignore)
    scanned = table[table["group"] == "scan"]
    print(f"{scanned['term'].nunique()} different terms in the scanned articles\n")

    # Table 2 only uses the periods with enough articles for a share to mean something
    all_periods = df.loc[df["group"] == "scan", "period"].value_counts().sort_index()
    period_sizes = all_periods[all_periods >= args.min_period_articles]
    print(f"Scanned articles per period:\n{all_periods.to_string()}")
    left_out = [period for period in all_periods.index if period not in period_sizes.index]
    if left_out:
        print(f"Left out of table 2, fewer than {args.min_period_articles} articles: {', '.join(left_out)}")

    latest = df["published"].max()
    cutoff = latest - pd.DateOffset(months=args.recent_months)

    tables = {"distinctive": distinctive(table, sizes, args.min_articles)}
    if len(period_sizes) >= 2:
        tables["rising"] = rising(scanned[scanned["period"].isin(period_sizes.index)], period_sizes, args.min_articles)
    else:
        print("Table 2 not written: it needs at least two periods with enough articles.")
    tables["weak_signals"] = weak_signals(scanned, cutoff, args.weak_min, args.weak_max, args.min_sources)

    titles = {
        "distinctive": f"1. Distinctive terms (in at least {args.min_articles} scanned articles), biggest difference of share first",
        "rising": f"2. Share per period (in at least {args.min_articles} scanned articles), biggest rise first",
        "weak_signals": (
            f"3. Terms in {args.weak_min} to {args.weak_max} articles, from at least {args.min_sources} sources, "
            f"first seen after {cutoff:%Y-%m-%d} (newest article: {latest:%Y-%m-%d})"
        ),
    }
    Path(args.out).parent.mkdir(parents=True, exist_ok=True)
    written = []
    for name, result in tables.items():
        path = f"{args.out}_{name}.csv"
        result.to_csv(path, index=False)
        written.append(f"{len(result)} terms -> {path}")
        print(f"\n{titles[name]}:")
        shown = result.drop(columns=["example_url"], errors="ignore").head(args.show)
        print(shown.to_string(index=False, max_colwidth=60) if len(shown) else "(none)")

    print("\nWrote:")
    print("\n".join(written))

### Options of step 6

In [ ]:
import shlex  # cuts the OPTIONS text into words, the way the command line does

# The options, written as on the command line. "" means every default.
# "--help" lists all of them (and then stops the cell, which is normal).
# Examples from the script:
#   OPTIONS = '--period half'
#   OPTIONS = '--scan others --text header'
#   OPTIONS = '--recent-months 6 --weak-max 8'
#   OPTIONS = '--ignore google,alphabet --show 30'
OPTIONS = ""

### Run step 6

In [ ]:
main()